# فصل ۷: ساخت برنامه‌های چت
## شروع سریع با Azure OpenAI API


## نمای کلی
این دفترچه یادداشت از [مخزن نمونه‌های Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) اقتباس شده است که شامل دفترچه‌هایی است که همچنین به سرویس [OpenAI](notebook-openai.ipynb) دسترسی دارند.

API پایتون OpenAI با Azure OpenAI نیز کار می‌کند، با چند تغییر کوچک. برای اطلاعات بیشتر در مورد تفاوت‌ها اینجا را ببینید: [نحوه سوئیچ بین نقاط پایانی OpenAI و Azure OpenAI با پایتون](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

برای مثال‌های سریع بیشتر لطفاً به مستندات رسمی [شروع سریع Azure OpenAI](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) مراجعه کنید


## فهرست مطالب  

[بررسی اجمالی](#overview)  
[شروع به کار با سرویس Azure OpenAI](#getting-started-with-azure-openai-service)  
[ساخت اولین درخواست](#build-your-first-prompt)  

[موارد استفاده](#use-cases)    
[1. خلاصه کردن متن](#summarize-text)  
[2. دسته‌بندی متن](#classify-text)  
[3. تولید نام‌های جدید محصول](#generate-new-product-names)  
[4. تنظیم دقیق دسته‌بند](#fine-tune-a-classifier)  
[5. جاسازی‌ها](#embeddings)

[مراجع](#references)


### شروع کار با سرویس Azure OpenAI  

مشتریان جدید باید برای [دریافت دسترسی](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) به سرویس Azure OpenAI درخواست دهند.  
پس از تأیید، مشتریان می‌توانند وارد پورتال Azure شوند، یک منبع سرویس Azure OpenAI ایجاد کنند و با استفاده از استودیو شروع به آزمایش مدل‌ها کنند  

[منبع عالی برای شروع سریع](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)  


### اولین درخواست خود را بسازید  
این تمرین کوتاه یک معرفی پایه برای ارسال درخواست‌ها به مدل OpenAI برای یک کار ساده «خلاصه‌سازی» ارائه می‌دهد.


**مراحل**:  
1. نصب کتابخانه OpenAI در محیط پایتون خود  
2. بارگذاری کتابخانه‌های کمکی استاندارد و تنظیم مدارک امنیتی معمول OpenAI برای سرویس OpenAI که ایجاد کرده‌اید  
3. انتخاب یک مدل برای کار خود  
4. ایجاد یک درخواست ساده برای مدل  
5. ارسال درخواست خود به API مدل!


### ۱. نصب OpenAI


  > [!NOTE] این مرحله در صورتی که این نوت‌بوک را در Codespaces یا داخل یک Devcontainer اجرا کنید، لازم نیست


In [ ]:
%pip install openai python-dotenv

### ۲. وارد کردن کتابخانه‌های کمکی و ایجاد نمونه احراز هویت‌ها


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### ۳. یافتن مدل مناسب  
مدل‌هایی مانند GPT-4o و GPT-4o mini می‌توانند زبان طبیعی را درک کرده و تولید کنند. Azure OpenAI در Microsoft Foundry مجموعه‌ای از قابلیت‌های مدل را ارائه می‌دهد که هر کدام با سطوح مختلف قدرت و سرعت برای وظایف مختلف مناسب هستند. 

[Azure OpenAI در Microsoft Foundry مدل‌ها](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## ۴. طراحی پرامپت  

«جادوی مدل‌های بزرگ زبان این است که با آموزش دیدن برای کمینه کردن خطای پیش‌بینی بر روی حجم وسیعی از متن‌ها، مدل‌ها در نهایت مفاهیمی را می‌آموزند که برای این پیش‌بینی‌ها مفید هستند. برای مثال، آنها مفاهیمی مانند»(1):

* چگونه املا را انجام دهند
* نحوه کارکرد دستور زبان
* چگونه بازنویسی کنند
* چگونه به سوالات پاسخ دهند
* چگونه مکالمه برقرار کنند
* چگونه به زبان‌های متعدد بنویسند
* چگونه کدنویسی کنند
* و غیره.

#### چگونه مدل بزرگ زبان را کنترل کنیم  
«از میان تمام ورودی‌ها به یک مدل بزرگ زبان، بدون شک مهم‌ترین ورودی، پرامپت متنی است»(1).

مدل‌های بزرگ زبان می‌توانند به چند روش پرامپت شوند تا خروجی تولید کنند:

- دستورالعمل: به مدل بگویید چه می‌خواهید
- تکمیل: موجب شوید مدل ابتدای آنچه می‌خواهید را کامل کند
- نمایش: به مدل نشان دهید چه می‌خواهید، با یکی از این موارد:
- چند نمونه در پرامپت
- صدها یا هزاران نمونه در یک مجموعه داده آموزش دقیق



#### سه دستورالعمل پایه برای ایجاد پرامپت وجود دارد:

**نشان دهید و بیان کنید**. واضح کنید چه می‌خواهید از طریق دستورالعمل‌ها، نمونه‌ها یا ترکیبی از هر دو. اگر می‌خواهید مدل یک لیست از آیتم‌ها را به ترتیب حروف الفبا رتبه‌بندی کند یا یک پاراگراف را بر اساس احساسات طبقه‌بندی کند، به آن نشان دهید که این خواسته شما است.

**داده‌های با کیفیت ارائه دهید**. اگر سعی دارید یک دسته‌بند بسازید یا مدل را برای پیروی از یک الگو آموزش دهید، مطمئن شوید نمونه‌های کافی وجود دارد. حتماً نمونه‌های خود را بازبینی کنید — معمولاً مدل آن‌قدر هوشمند است که از خطاهای املایی ساده عبور کند و پاسخ دهد، اما ممکن است فرض کند این عمدی است و این می‌تواند روی پاسخ تأثیر بگذارد.

**تنظیمات خود را بررسی کنید.** تنظیمات temperature و top_p کنترل می‌کنند که مدل چقدر در تولید پاسخ قطعی باشد. اگر از مدل درخواست پاسخی را دارید که فقط یک جواب درست دارد، باید این تنظیمات را پایین تنظیم کنید. اگر به دنبال پاسخ‌های متنوع‌تر هستید، ممکن است بخواهید آنها را بالاتر تنظیم کنید. اشتباه شماره یک مردم هنگام کار با این تنظیمات این است که فرض می‌کنند این‌ها کنترل «هوشمندی» یا «خلاقیت» هستند.


منبع: https://learn.microsoft.com/azure/ai-foundry/openai/overview


تصویر در حال ساخت اولین متن فرمان شماست!


### ۵. ارسال کنید!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### همان تماس را دوباره تکرار کنید، نتایج چگونه با هم مقایسه می‌شوند؟


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## خلاصه کردن متن  
#### چالش  
متن را با اضافه کردن «tl;dr:» در انتهای یک بخش متنی خلاصه کنید. توجه کنید که مدل چگونه بدون دستورالعمل‌های اضافی قادر است تعدادی از وظایف را انجام دهد. شما می‌توانید با پرسش‌های توصیفی‌تر از tl;dr رفتار مدل را تغییر داده و خلاصه‌سازی دریافتی خود را سفارشی کنید(3).  

کارهای اخیر نشان داده است که پیش‌آموزش روی یک مجموعه بزرگ متنی و سپس تنظیم دقیق روی یک وظیفه خاص، پیشرفت‌های قابل توجهی در بسیاری از وظایف و معیارهای پردازش زبان طبیعی به همراه دارد. اگرچه ساختار معمولاً مستقل از کار است، این روش هنوز به داده‌های تنظیم دقیق مخصوص کار با هزاران یا ده‌ها هزار نمونه نیاز دارد. در مقابل، انسان‌ها معمولاً می‌توانند فقط با چند نمونه یا دستورالعمل‌های ساده یک وظیفه زبانی جدید را انجام دهند - امری که سیستم‌های پردازش زبان طبیعی فعلی هنوز تا حد زیادی در انجام آن مشکل دارند. در اینجا نشان می‌دهیم که افزایش اندازه مدل‌های زبانی به طور قابل توجهی عملکرد چندنمونه‌ای مستقل از کار را بهبود می‌بخشد، و گاهی حتی به رقابت با روش‌های تنظیم دقیق پیشرفته پیشین می‌رسد.  



خلاصه  


# تمرین‌هایی برای چندین مورد کاربرد  
1. خلاصه کردن متن  
2. دسته‌بندی متن  
3. تولید نام‌های جدید محصول  
4. ایجاد جاسازی‌ها  
5. تنظیم دقیق یک دسته‌بندی‌کننده  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## طبقه‌بندی متن  
#### چالش  
آیتم‌ها را در دسته‌بندی‌هایی که در زمان استنتاج ارائه می‌شوند، طبقه‌بندی کنید. در مثال زیر، هم دسته‌ها و هم متن برای طبقه‌بندی را در درخواست ارائه می‌دهیم (*playground_reference). 

استعلام مشتری: سلام، یکی از کلیدهای صفحه‌کلید لپ‌تاپ من اخیراً شکسته و به یک جایگزین نیاز دارم:

دسته‌بندی شده:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## تولید نام‌های جدید برای محصول
#### چالش
ایجاد نام‌های محصول از کلمات نمونه. در اینجا اطلاعاتی درباره محصولی که قرار است برای آن نام تولید کنیم در پرامپت گنجانده‌ایم. همچنین یک نمونه مشابه ارائه داده‌ایم تا الگویی که می‌خواهیم دریافت کنیم نشان دهیم. ما همچنین مقدار دما را بالا تنظیم کرده‌ایم تا تصادفی بودن و پاسخ‌های نوآورانه بیشتری را افزایش دهیم.

شرح محصول: دستگاه ساخت میلک‌شیک خانگی
کلمات پایه: سریع، سالم، جمع‌وجور.
نام‌های محصول: HomeShaker، Fit Shaker، QuickShake، Shake Maker

شرح محصول: یک جفت کفش که می‌تواند به هر اندازه پا بخورد.
کلمات پایه: سازگار، مناسب، همه‌سایز.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## جاسازی‌ها
این بخش نشان می‌دهد که چگونه می‌توان جاسازی‌ها را بازیابی کرد و شباهت‌ها بین کلمات، جملات و اسناد را یافت. برای اجرای دفترچه‌های زیر، باید مدلی را که از `text-embedding-ada-002` به عنوان مدل پایه استفاده می‌کند، مستقر کرده و نام استقرار آن را در فایل .env با استفاده از متغیر `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` تنظیم کنید.


### رده‌بندی مدل - انتخاب مدل توکارسازی

**رده‌بندی مدل**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (خانواده توکارسازی)  
{capability} --> ada             (تمام مدل‌های توکارسازی دیگر در سال ۲۰۲۴ بازنشسته خواهند شد)  
{input-type} --> n/a             (تنها برای مدل‌های جستجو مشخص شده است)  
{identifier} --> 002             (نسخه ۰۰۲)  

model = 'text-embedding-ada-002'


  > [!NOTE] اگر این دفترچه یادداشت را در Codespaces یا داخل یک Devcontainer اجرا کنید، مرحله زیر لازم نیست


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## مقایسه مقاله از مجموعه داده خبری روزانه سی‌ان‌ان
منبع: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# منابع  
- [Microsoft Foundry - مدل‌های Azure OpenAI](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [پورتال Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# برای دریافت کمک بیشتر  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# مشارکت‌کنندگان
* لوئیس لی  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
